# Review Usability Evaluation Reports
This notebook allows you to browse and review the pre-generated usability evaluation reports without requiring a Google AI API Key. It loads the results of the evaluation approach presented in the paper.

The results are divided into three sections, representing the pipeline stages:
1. **Heuristic Evaluation**: Raw findings from the initial MLLM evaluation for each task and heuristic.
2. **Aggregated Report**: Consolidated issues after merging overlapping findings across heuristics.
3. **Severity Ranking**: The final prioritized list of usability problems re-ranked by the MLLM.


In [ ]:
import json
import os
import pandas as pd
from IPython.display import display, Markdown

# Choose the dataset to review
DATA_PATH = "../data/KnowledgeCheckR/"
# DATA_PATH = "../data/EventHelpR/"

REPORTS_PATH = os.path.join(DATA_PATH, "reports")


## 1. Heuristic Evaluation
This section shows the raw results of the heuristic evaluation, where each heuristic was evaluated independently for each task.


In [ ]:
# Load all evaluation_*.json files
evaluation_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("evaluation_") and f.endswith(".json")])
all_evaluation_issues = []

for file in evaluation_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for entry in data:
            heuristic = entry.get("heuristic", "Unknown")
            task_id = entry.get("id", "Unknown").split("_")[0:3]
            task_id = "_".join(task_id) if isinstance(task_id, list) else task_id
            for issue in entry.get("issues", []):
                issue['source_file'] = file
                issue['heuristic'] = heuristic
                issue['task_id'] = task_id
                all_evaluation_issues.append(issue)

df_eval = pd.DataFrame(all_evaluation_issues)
if not df_eval.empty:
    display(Markdown(f"### Raw Evaluation Issues for {DATA_PATH.split('/')[-2]}"))
    # Grouping by heuristic to show counts
    display(Markdown("#### Issues Count by Heuristic"))
    display(df_eval.groupby('heuristic').size().to_frame('count').sort_values('count', ascending=False))
    
    display(Markdown("#### Raw Issues"))
    display(df_eval[['task_id', 'severity', 'heuristic', 'description']])
else:
    print("No raw evaluation reports found.")


## 2. Aggregated Report
This section shows the consolidated issues after semantic clustering and summarization. Overlapping findings across heuristics are merged.


In [ ]:
# Load all report_*.json files (excluding ranking reports)
report_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("report_") and f.endswith(".json")])
all_aggregated_issues = []

for file in report_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for task_id, issues in data.items():
            for issue in issues:
                issue['source_file'] = file
                issue['task_id'] = task_id
                # Heuristics is a list here
                if 'heuristics' in issue:
                    issue['heuristic_list'] = ", ".join(issue['heuristics'])
                all_aggregated_issues.append(issue)

df_agg = pd.DataFrame(all_aggregated_issues)
if not df_agg.empty:
    display(Markdown(f"### Aggregated Issues for {DATA_PATH.split('/')[-2]}"))
    # Show issues grouped by Task ID for better context
    for task in df_agg['task_id'].unique():
        display(Markdown(f"#### Task: {task}"))
        task_df = df_agg[df_agg['task_id'] == task]
        display(task_df[['severity', 'heuristic_list', 'description']])
else:
    print("No aggregated reports found.")


## 3. Severity Ranking
This section shows the final prioritized list of usability problems, re-ranked by an LLM based on their overall impact.


In [ ]:
# Load all ranking_report_*.json files
ranking_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("ranking_report_") and f.endswith(".json")])
all_ranked_issues = []

for file in ranking_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for task_id, issues in data.items():
            for issue in issues:
                issue['source_file'] = file
                issue['task_id'] = task_id
                all_ranked_issues.append(issue)

df_ranked = pd.DataFrame(all_ranked_issues)
if not df_ranked.empty:
    display(Markdown(f"### Final Ranked Issues for {DATA_PATH.split('/')[-2]}"))
    # Sort by Task ID if necessary, though they are usually in order
    for task_id in df_ranked['task_id'].unique():
        display(Markdown(f"#### Ranked Issues for Task: {task_id}"))
        task_df = df_ranked[df_ranked['task_id'] == task_id].reset_index(drop=True)
        # Show rank index explicitly starting from 1
        task_df.index += 1
        display(task_df[['severity', 'description', 'rank_justification']])
else:
    print("No ranking reports found.")
